In [ ]:
%load_ext autoreload
%autoreload 2

In [5]:
import sqlite3
import scipy
import json
from estnltk import Text
from estnltk.taggers import VabamorfAnalyzer

In [7]:
SOURCE_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
CONFLICT_DB = "syntax_morphology_conflicts.db"
SENTENCES_DB = "v33_koondkorpus_sentences_sentences_20250220-130121.db"
UNRESOLVED_IDX = "unresolved_idx.db"
RESULT_ONE_POSSIBLE_CASE = "one_possible_case.db"
RESULT_ADVMOD = "advmod.db"
RESULT_ADVMOD_DEPREL_CONFLICT = "advmod_deprel_conflict.db"
RESULT_NSUBJ_PART = "nsubj_part.db"
RESULT_FIRST_POSITION = "first_position.db"
RESULT_UPPERCASE = "uppercase.db"

In [8]:
morph_analyzer = VabamorfAnalyzer()

In [12]:
def get_n_possible_cases(field, n_cases):
    return len(json.loads(field).split()) == n_cases

def starts_with_uppercase(field):
    return field.istitle()

def get_n_vabamorf_cases(phrase_field, phrase_root_lemma, n_cases):
    # hetkel pole veel parandanud ära, et kasutaks analyze_token funktsiooni
    form_mapping = {
      'sg p': 'part',
      'sg g': 'gen',
      'sg n': 'nom',
      'pl p': 'part',
      'pl g': 'gen',
      'pl n': 'nom',
      'sg el': 'el',
      'pl el': 'el',
      'sg tr': 'tr',
      'pl tr': 'tr',
      'sg in': 'in',
      'pl in': 'in',
      'sg es': 'es',
      'pl es': 'es',
      'sg ter': 'ter',
      'pl ter': 'ter',
      'sg kom': 'kom',
      'pl kom': 'kom',
      'sg ill': 'ill',
      'pl ill': 'ill',
      'sg abl': 'abl',
      'pl abl': 'abl',
      'sg ad': 'ad',
      'pl ad': 'ad',
      'sg all': 'all',
      'pl all': 'all',
      'adt':'adt',
      '':'',
      '?': '?'
  }
    phrase_txt = Text(str(phrase_field)).tag_layer("words")
    lemma_form = {}
    for idx, word in enumerate(phrase_txt.words):
        analysis = morph_analyzer.analyze_token(word.text)
        for idx2, a in enumerate(analysis):
            if a["lemma"] == phrase_root_lemma and a["partofspeech"] != "V":
                lemma_form[a["lemma"]] = []
    for idx, word in enumerate(phrase_txt.words):
        analysis = morph_analyzer.analyze_token(word.text)
        for idx2, a in enumerate(analysis):
            if a["lemma"] == phrase_root_lemma and a["partofspeech"] != "V":
                lemma_form[a["lemma"]]+=[form_mapping[a["form"]]]
    n_analyses = []
    for key, value in lemma_form.items():
        n_analyses.append(len(set(value)))
    return n_cases in n_analyses

In [ ]:
# Esimesena juhud, mille kohta vabamorf ütleb, et üks kääne
# Sain lõpuks pihta, aga ei saanud veel implementeeritud, et mida rektsiooni järgi "ainuvõimaliku" käände all mõeldud on
# Endiselt ei tea päris kindlalt, mis asi "possible_cases" väli on
# Hetkel funktsioon annab mingi errori, vaja debuggida

con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("sql_get_n_vabamorf_cases", 3, get_n_vabamorf_cases)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ONE_POSSIBLE_CASE}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.one_possible_case
""")

cur.execute("""
    CREATE TABLE result.one_possible_case AS
    SELECT 
            tbl1.id,
            tbl1.pattern_id,
            tbl1.sentence_id,
            tbl1.verb_loc,
            tbl1.compound_loc,
            tbl1.phrase_root_loc,
            tbl1.verb_phrase_loc,
            tbl1.phrase_case,
            tbl1.phrase_deprel,
            tbl1.verb,
            tbl1.verb_compound,
            tbl1.phrase,
            tbl1.phrase_root_lemma,
            tbl1.current_analysis,
            tbl1.current_case,
            tbl1.possible_cases,
            tbl2.sentence
    FROM
            (
                SELECT
                    *
                FROM
                    syntax_morphology_conflicts
                
                WHERE 
                    (verb, verb_compound) IN 
                    (
                        SELECT verb, verb_compound
                        FROM syntax_morphology_conflicts
                        GROUP BY verb, verb_compound
                        HAVING COUNT(DISTINCT phrase_case) = 1
                    )
                AND
                    sql_get_n_vabamorf_cases(phrase, phrase_root_lemma, 1)
                ) AS tbl1
        INNER JOIN
            (
                SELECT
                    id,
                    text AS sentence
                FROM
                    sents.sentences
                ) AS tbl2
            ON
                tbl1.sentence_id = tbl2.id          
    """)

cur.execute("""
    DROP TABLE IF EXISTS idxs.unresolved_idx
    """)

cur.execute("""
    CREATE TABLE idxs.unresolved_idx
    AS
    SELECT
        tbl1.id AS idx
    FROM
        syntax_morphology_conflicts AS tbl1
    LEFT JOIN
        result.one_possible_case AS tbl2
    ON 
        tbl1.id=tbl2.id
    WHERE 
        tbl2.id IS NULL
""")


con.close()

OperationalError: user-defined function raised exception

In [ ]:
# Teiseks juhud, kus deprel on advmod/advcl
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("sql_get_n_possible_cases", 2, get_n_possible_cases)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ADVMOD}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.advmod
""")

cur.execute("""
    CREATE TABLE result.advmod AS
    SELECT 
            tbl1.id,
            tbl1.pattern_id,
            tbl1.sentence_id,
            tbl1.verb_loc,
            tbl1.compound_loc,
            tbl1.phrase_root_loc,
            tbl1.verb_phrase_loc,
            tbl1.phrase_case,
            tbl1.phrase_deprel,
            tbl1.verb,
            tbl1.verb_compound,
            tbl1.phrase,
            tbl1.phrase_root_lemma,
            tbl1.current_analysis,
            tbl1.current_case,
            tbl1.possible_cases,
            tbl2.sentence
    FROM
            (
                SELECT
                    *
                FROM
                    syntax_morphology_conflicts AS conf
                INNER JOIN
                    idxs.unresolved_idx AS tmp
                ON
                    conf.id=tmp.idx
                WHERE
                    phrase_deprel="advmod" OR phrase_deprel="advcl"
                AND
                    sql_get_n_possible_cases(conf.possible_cases, 2)
                AND 
                    instr(conf.possible_cases, "null") > 0
            ) AS tbl1
    INNER JOIN
            (
                SELECT
                    id,
                    text AS sentence
                FROM
                    sents.sentences
            ) AS tbl2
        ON
            tbl1.sentence_id = tbl2.id          
            """)


cur.execute("""
    DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
    CREATE TABLE idxs.new_unresolved_idx
    AS
    SELECT
        tbl1.idx
    FROM
        idxs.unresolved_idx AS tbl1
    LEFT JOIN
        result.advmod AS tbl2
    ON 
        tbl1.idx=tbl2.id
    WHERE 
        tbl2.id IS NULL
""")

cur.execute("""
    DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
    ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [ ]:
# Kolmandaks juhud, kus juursõna käitub kui määrus, aga deprel pole advmod/advcl
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("sql_get_n_possible_cases", 2, get_n_possible_cases)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_ADVMOD_DEPREL_CONFLICT}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.advmod_wrong_deprel
""")

cur.execute("""
    CREATE TABLE result.advmod_wrong_deprel AS
    SELECT 
            tbl1.id,
            tbl1.pattern_id,
            tbl1.sentence_id,
            tbl1.verb_loc,
            tbl1.compound_loc,
            tbl1.phrase_root_loc,
            tbl1.verb_phrase_loc,
            tbl1.phrase_case,
            tbl1.phrase_deprel,
            tbl1.verb,
            tbl1.verb_compound,
            tbl1.phrase,
            tbl1.phrase_root_lemma,
            tbl1.current_analysis,
            tbl1.current_case,
            tbl1.possible_cases,
            tbl2.sentence
    FROM
            (
            SELECT
                *
            FROM
                syntax_morphology_conflicts AS conf
            INNER JOIN
                idxs.unresolved_idx AS tmp
            ON
                conf.id=tmp.idx
            WHERE
                sql_get_n_possible_cases(conf.possible_cases, 2)
            AND 
                instr(conf.possible_cases, "null") > 0
            ) AS tbl1
    INNER JOIN
            (
            SELECT
                id,
                text AS sentence
            FROM
                sents.sentences
            ) AS tbl2
    ON
            tbl1.sentence_id = tbl2.id          
            """)


cur.execute("""
    DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
    CREATE TABLE idxs.new_unresolved_idx
    AS
    SELECT
        tbl1.idx
    FROM
        idxs.unresolved_idx AS tbl1
    LEFT JOIN
        result.advmod_wrong_deprel AS tbl2
    ON 
        tbl1.idx=tbl2.id
    WHERE 
        tbl2.id IS NULL
""")

cur.execute("""
    DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
    ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [ ]:
# Neljandaks juhud, kus deprel nsubj ja võimalike käänete hulgas part
# uuesti Hendriku tööd vaadates tundub, et ikkagi possible_cases seest tuleb otsida, mitte rektsioonil põhinevat käänet vaadata
# KÜSIMUS: Kas loogika on, et kuigi ühestamisel saab õige käände (nom), siis võimalike variantide hulgas partitiivi nähes
# teame, et tegemist on sõnadega, milles esineb üleüldiselt veaohtlik vormihomonüümia?
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_NSUBJ_PART}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.nsubj_part
""")

cur.execute("""
    CREATE TABLE result.nsubj_part AS
    SELECT 
            tbl1.id,
            tbl1.pattern_id,
            tbl1.sentence_id,
            tbl1.verb_loc,
            tbl1.compound_loc,
            tbl1.phrase_root_loc,
            tbl1.verb_phrase_loc,
            tbl1.phrase_case,
            tbl1.phrase_deprel,
            tbl1.verb,
            tbl1.verb_compound,
            tbl1.phrase,
            tbl1.phrase_root_lemma,
            tbl1.current_analysis,
            tbl1.current_case,
            tbl1.possible_cases,
            tbl2.sentence
    FROM
            (
            SELECT
                *
            FROM
                syntax_morphology_conflicts AS conf
            INNER JOIN
                idxs.unresolved_idx AS tmp
            ON
                conf.id=tmp.idx
            WHERE
                phrase_deprel = "nsubj"
            AND
                instr(possible_cases, "part") > 0
            ) AS tbl1
    INNER JOIN
            (
            SELECT
                id,
                text AS sentence
            FROM
                sents.sentences
            ) AS tbl2
    ON
            tbl1.sentence_id = tbl2.id          
            """)

cur.execute("""
    DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
    CREATE TABLE idxs.new_unresolved_idx
    AS
    SELECT
        tbl1.idx
    FROM
        idxs.unresolved_idx AS tbl1
    LEFT JOIN
        result.nsubj_part AS tbl2
    ON 
        tbl1.idx=tbl2.id
    WHERE 
        tbl2.id IS NULL
""")

cur.execute("""
    DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
    ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [ ]:
# Viiendaks juhud, kus juursõnaks fraasi esimene sõna, on eeldus, et valdavalt peaks see olema nsubj
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_FIRST_POSITION}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.first_position
""")

cur.execute("""
    CREATE TABLE result.first_position AS
    SELECT 
            tbl1.id,
            tbl1.pattern_id,
            tbl1.sentence_id,
            tbl1.verb_loc,
            tbl1.compound_loc,
            tbl1.phrase_root_loc,
            tbl1.verb_phrase_loc,
            tbl1.phrase_case,
            tbl1.phrase_deprel,
            tbl1.verb,
            tbl1.verb_compound,
            tbl1.phrase,
            tbl1.phrase_root_lemma,
            tbl1.current_analysis,
            tbl1.current_case,
            tbl1.possible_cases,
            tbl2.sentence
    FROM
            (
            SELECT
                *
            FROM
                syntax_morphology_conflicts AS conf
            INNER JOIN
                idxs.unresolved_idx AS tmp
            ON
                conf.id = tmp.idx
            WHERE
                phrase_root_loc=1
            ) AS tbl1
    INNER JOIN
            (
            SELECT
                id,
                text AS sentence
            FROM
                sents.sentences
            ) AS tbl2
    ON
            tbl1.sentence_id = tbl2.id          
            """)

cur.execute("""
    DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
    CREATE TABLE idxs.new_unresolved_idx
    AS
    SELECT
        tbl1.idx
    FROM
        idxs.unresolved_idx AS tbl1
    LEFT JOIN
        idxs.first_position AS tbl2
    ON 
        tbl1.idx=tbl2.id
    WHERE 
        tbl2.id IS NULL
""")

cur.execute("""
    DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
    ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()

In [ ]:
# Kuuendaks pärisnimed
con = sqlite3.connect(f"{SOURCE_DATA_PATH}{CONFLICT_DB}")

con.create_function("sql_starts_with_uppercase", 1, starts_with_uppercase)

cur = con.cursor()

cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{SENTENCES_DB}" AS sents')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{RESULT_UPPERCASE}" AS result')
cur.execute(f'ATTACH DATABASE "{SOURCE_DATA_PATH}{UNRESOLVED_IDX}" AS idxs')

cur.execute("""
DROP TABLE IF EXISTS result.uppercase
""")

cur.execute("""
    CREATE TABLE result.uppercase AS
    SELECT 
            tbl1.id,
            tbl1.pattern_id,
            tbl1.sentence_id,
            tbl1.verb_loc,
            tbl1.compound_loc,
            tbl1.phrase_root_loc,
            tbl1.verb_phrase_loc,
            tbl1.phrase_case,
            tbl1.phrase_deprel,
            tbl1.verb,
            tbl1.verb_compound,
            tbl1.phrase,
            tbl1.phrase_root_lemma,
            tbl1.current_analysis,
            tbl1.current_case,
            tbl1.possible_cases,
            tbl2.sentence
    FROM
            (
            SELECT
                *
            FROM
                syntax_morphology_conflicts AS conf
            INNER JOIN
                idxs.unresolved_idx AS tmp
            ON
                conf.id = tmp.idx
            WHERE
                sql_starts_with_uppercase(phrase_root_lemma)
            ) AS tbl1
    INNER JOIN
            (
            SELECT
                id,
                text AS sentence
            FROM
                sents.sentences
            ) AS tbl2
    ON
            tbl1.sentence_id = tbl2.id          
            """)

cur.execute("""
    DROP TABLE IF EXISTS idxs.new_unresolved_idx
""")

cur.execute("""
    CREATE TABLE idxs.new_unresolved_idx
    AS
    SELECT
        tbl1.idx
    FROM
        idxs.unresolved_idx AS tbl1
    LEFT JOIN
        idxs.uppercase AS tbl2
    ON 
        tbl1.idx=tbl2.id
    WHERE 
        tbl2.id IS NULL
""")

cur.execute("""
    DROP TABLE idxs.unresolved_idx
""")

cur.execute("""
    ALTER TABLE idxs.new_unresolved_idx RENAME TO unresolved_idx
""")

con.close()